# Lab Session: Knowledge Base Construction, Alignment & Expansion
## TP4 — Daniel ORDUYREY · Yanis NAITKACI

This notebook implements a complete pipeline for constructing a private Knowledge Base (KB) in RDF,
its alignment to Wikidata, and its controlled expansion via SPARQL queries.

**Volume targets (PDF guidelines)**

| Metric | Min | Max |
|---|---|---|
| Triplets | 50,000 | 200,000 |
| Entities | 5,000  | 30,000  |
| Relations | 50    | **200 max** |

## 0 — Installing Dependencies

In [1]:
import subprocess, sys

def pip_install(pkg):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])

for pkg in ['rdflib', 'requests', 'tqdm', 'pandas']:
    pip_install(pkg)

print('Dependances installees.')


Dependances installees.


## Step 1 — Initial Private KB Construction

**Why this step?**
We are building the **TBox** (ontological schema: classes, properties, domains/ranges)
and the **ABox** (real data) from a dataset of researchers and Nobel prizes.
The data is loaded via Pandas, representing the extracted output from Lab 1.

A `uri_safe()` function cleans all strings before injecting them into URIs,
avoiding any illegal characters (spaces, parentheses, quotes, dashes).

In [15]:
import re
import pandas as pd
from rdflib import Graph, Namespace, URIRef, Literal
from rdflib.namespace import RDF, RDFS, OWL, XSD

# ── Namespaces ────────────────────────────────────────────────────────────
g = Graph()
EX = Namespace('http://example.org/private#')
g.bind('ex',   EX)
g.bind('rdfs', RDFS)
g.bind('owl',  OWL)

# ── Utilitaire : nettoyage URI ────────────────────────────────────────────
def uri_safe(text: str) -> str:
    text = str(text).strip()
    text = re.sub(r'[\s\(\)\"\',;:/\\\-]', '_', text)
    text = re.sub(r'_+', '_', text)
    return text.strip('_')

# ── Correspondance NER -> Classe ontologique ──────────────────────────────
LABEL_TO_CLASS = {
    'PERSON':    EX.Researcher,
    'ORG':       EX.Organization,
    'AWARD':     EX.Award,
    'INSTITUTE': EX.ResearchInstitute,
}

# ── TBox : Schema Ontologique ─────────────────────────────────────────────
g.add((EX.Researcher,        RDF.type,        OWL.Class))
g.add((EX.Organization,      RDF.type,        OWL.Class))
g.add((EX.Award,             RDF.type,        OWL.Class))
g.add((EX.ResearchInstitute, RDF.type,        OWL.Class))
g.add((EX.University,        RDF.type,        OWL.Class))
g.add((EX.ResearchInstitute, RDFS.subClassOf, EX.Organization))
g.add((EX.University,        RDFS.subClassOf, EX.Organization))

props = [
    (EX.wonAward,       EX.Researcher, EX.Award),
    (EX.affiliatedWith, EX.Researcher, EX.Organization),
    (EX.employedBy,     EX.Researcher, EX.Organization),
    (EX.fieldOfStudy,   EX.Researcher, None),
    (EX.nationality,    EX.Researcher, None),
]
for prop, domain, rng in props:
    g.add((prop, RDF.type,    OWL.ObjectProperty))
    g.add((prop, RDFS.domain, domain))
    if rng:
        g.add((prop, RDFS.range, rng))

# ── ABox : Donnees reelles chargees via Pandas ────────────────────────────
raw_data = [
    {'researcher':'Marie Curie',           'award':'Nobel Prize in Physics',      'affiliation':'Sorbonne University',                'field':'Physics',              'nationality':'French'},
    {'researcher':'Albert Einstein',        'award':'Nobel Prize in Physics',      'affiliation':'Institute for Advanced Study',        'field':'Theoretical Physics',  'nationality':'German'},
    {'researcher':'Rosalind Franklin',      'award':None,                          'affiliation':'Kings College London',                'field':'Biophysics',           'nationality':'British'},
    {'researcher':'Alain Aspect',           'award':'Nobel Prize in Physics',      'affiliation':'Universite Paris Saclay',             'field':'Quantum Physics',      'nationality':'French'},
    {'researcher':'Richard Feynman',        'award':'Nobel Prize in Physics',      'affiliation':'California Institute of Technology',  'field':'Quantum Mechanics',    'nationality':'American'},
    {'researcher':'Niels Bohr',             'award':'Nobel Prize in Physics',      'affiliation':'University of Copenhagen',            'field':'Atomic Physics',       'nationality':'Danish'},
    {'researcher':'Werner Heisenberg',      'award':'Nobel Prize in Physics',      'affiliation':'University of Leipzig',               'field':'Quantum Mechanics',    'nationality':'German'},
    {'researcher':'Max Planck',             'award':'Nobel Prize in Physics',      'affiliation':'Humboldt University Berlin',          'field':'Theoretical Physics',  'nationality':'German'},
    {'researcher':'Erwin Schrodinger',      'award':'Nobel Prize in Physics',      'affiliation':'University of Vienna',                'field':'Quantum Mechanics',    'nationality':'Austrian'},
    {'researcher':'Paul Dirac',             'award':'Nobel Prize in Physics',      'affiliation':'University of Cambridge',             'field':'Theoretical Physics',  'nationality':'British'},
    {'researcher':'Enrico Fermi',           'award':'Nobel Prize in Physics',      'affiliation':'University of Chicago',               'field':'Nuclear Physics',      'nationality':'Italian'},
    {'researcher':'Lise Meitner',           'award':None,                          'affiliation':'Kaiser Wilhelm Institute',            'field':'Nuclear Physics',      'nationality':'Austrian'},
    {'researcher':'Peter Higgs',            'award':'Nobel Prize in Physics',      'affiliation':'University of Edinburgh',             'field':'Particle Physics',     'nationality':'British'},
    {'researcher':'Steven Weinberg',        'award':'Nobel Prize in Physics',      'affiliation':'University of Texas',                 'field':'Particle Physics',     'nationality':'American'},
    {'researcher':'Abdus Salam',            'award':'Nobel Prize in Physics',      'affiliation':'Imperial College London',             'field':'Particle Physics',     'nationality':'Pakistani'},
    {'researcher':'Sheldon Glashow',        'award':'Nobel Prize in Physics',      'affiliation':'Harvard University',                  'field':'Particle Physics',     'nationality':'American'},
    {'researcher':'Murray Gell Mann',       'award':'Nobel Prize in Physics',      'affiliation':'California Institute of Technology',  'field':'Particle Physics',     'nationality':'American'},
    {'researcher':'Leon Lederman',          'award':'Nobel Prize in Physics',      'affiliation':'Fermi National Laboratory',           'field':'Particle Physics',     'nationality':'American'},
    {'researcher':'Carlo Rubbia',           'award':'Nobel Prize in Physics',      'affiliation':'CERN',                               'field':'Particle Physics',     'nationality':'Italian'},
    {'researcher':'Georges Charpak',        'award':'Nobel Prize in Physics',      'affiliation':'CERN',                               'field':'Particle Physics',     'nationality':'French'},
    {'researcher':'Dorothy Hodgkin',        'award':'Nobel Prize in Chemistry',    'affiliation':'University of Oxford',               'field':'X Ray Crystallography','nationality':'British'},
    {'researcher':'Gertrude Elion',         'award':'Nobel Prize in Physiology',   'affiliation':'Duke University',                    'field':'Pharmacology',         'nationality':'American'},
    {'researcher':'Ada Yonath',             'award':'Nobel Prize in Chemistry',    'affiliation':'Weizmann Institute',                 'field':'Structural Biology',   'nationality':'Israeli'},
    {'researcher':'Frances Arnold',         'award':'Nobel Prize in Chemistry',    'affiliation':'California Institute of Technology',  'field':'Biochemistry',         'nationality':'American'},
    {'researcher':'Jennifer Doudna',        'award':'Nobel Prize in Chemistry',    'affiliation':'University of California Berkeley',  'field':'Biochemistry',         'nationality':'American'},
    {'researcher':'Emmanuelle Charpentier', 'award':'Nobel Prize in Chemistry',    'affiliation':'Max Planck Institute',               'field':'Microbiology',         'nationality':'French'},
    {'researcher':'Linus Pauling',          'award':'Nobel Prize in Chemistry',    'affiliation':'California Institute of Technology',  'field':'Chemistry',            'nationality':'American'},
    {'researcher':'Frederick Sanger',       'award':'Nobel Prize in Chemistry',    'affiliation':'University of Cambridge',            'field':'Biochemistry',         'nationality':'British'},
    {'researcher':'Alan Turing',            'award':None,                          'affiliation':'University of Manchester',           'field':'Computer Science',     'nationality':'British'},
    {'researcher':'Grace Hopper',           'award':'Presidential Medal of Freedom','affiliation':'Yale University',                   'field':'Computer Science',     'nationality':'American'},
    {'researcher':'Ada Lovelace',           'award':None,                          'affiliation':'University of London',               'field':'Mathematics',          'nationality':'British'},
    {'researcher':'Barbara McClintock',     'award':'Nobel Prize in Physiology',   'affiliation':'Cold Spring Harbor Laboratory',      'field':'Genetics',             'nationality':'American'},
    {'researcher':'James Watson',           'award':'Nobel Prize in Physiology',   'affiliation':'Cold Spring Harbor Laboratory',      'field':'Genetics',             'nationality':'American'},
    {'researcher':'Francis Crick',          'award':'Nobel Prize in Physiology',   'affiliation':'Salk Institute',                    'field':'Molecular Biology',    'nationality':'British'},
    {'researcher':'Kary Mullis',            'award':'Nobel Prize in Chemistry',    'affiliation':'Cetus Corporation',                  'field':'Biochemistry',         'nationality':'American'},
    {'researcher':'George Smith',           'award':'Nobel Prize in Chemistry',    'affiliation':'University of Missouri',             'field':'Biochemistry',         'nationality':'American'},
    {'researcher':'Roger Tsien',            'award':'Nobel Prize in Chemistry',    'affiliation':'University of California San Diego', 'field':'Biochemistry',         'nationality':'American'},
    {'researcher':'Osamu Shimomura',        'award':'Nobel Prize in Chemistry',    'affiliation':'Marine Biological Laboratory',       'field':'Biochemistry',         'nationality':'Japanese'},
    {'researcher':'Martin Chalfie',         'award':'Nobel Prize in Chemistry',    'affiliation':'Columbia University',                'field':'Neuroscience',         'nationality':'American'},
    {'researcher':'John Fenn',              'award':'Nobel Prize in Chemistry',    'affiliation':'Yale University',                    'field':'Chemistry',            'nationality':'American'},
]

df = pd.DataFrame(raw_data)

for _, row in df.iterrows():
    person_uri = EX[uri_safe(row['researcher'])]
    org_uri    = EX[uri_safe(row['affiliation'])]

    g.add((person_uri, RDF.type,          EX.Researcher))
    g.add((person_uri, RDFS.label,        Literal(row['researcher'],  lang='en')))
    g.add((org_uri,    RDF.type,          EX.Organization))
    g.add((org_uri,    RDFS.label,        Literal(row['affiliation'], lang='en')))
    g.add((person_uri, EX.affiliatedWith, org_uri))
    g.add((person_uri, EX.fieldOfStudy,   Literal(row['field'],       lang='en')))
    g.add((person_uri, EX.nationality,    Literal(row['nationality'], lang='en')))

    if pd.notna(row['award']):
        award_uri = EX[uri_safe(row['award'])]
        g.add((award_uri,  RDF.type,    EX.Award))
        g.add((award_uri,  RDFS.label,  Literal(row['award'], lang='en')))
        g.add((person_uri, EX.wonAward, award_uri))

g.serialize(destination='initial_private_kb.ttl', format='turtle')
n_triplets = len(g)
n_entities = len(set(g.subjects()))
print(f'Graphe initial : {n_triplets} triplets, {n_entities} entites')
assert n_triplets >= 100, f'Seulement {n_triplets} triplets (min 100 requis)'
print('Graphe sauvegarde : initial_private_kb.ttl')


Graphe initial : 330 triplets, 87 entites
Graphe sauvegarde : initial_private_kb.ttl


## Step 2 — Entity Alignment (Entity Linking to Wikidata)

**Why this step?**
Each entity in our private KB (e.g., `ex:Marie_Curie`) is linked to its Wikidata equivalent (e.g., `wd:Q7186`) via an `owl:sameAs` triple. [cite_start]This alignment signifies that the local entity is identical to the one in the external Knowledge Base[cite: 98].

**Strategy**:
1. [cite_start]Extract the `rdfs:label` for each entity[cite: 98].
2. [cite_start]Query the Wikidata `wbsearchentities` API[cite: 98].
3. [cite_start]Use a **professional `User-Agent`**, which is mandatory to prevent Wikimedia from blocking the requests[cite: 98].
4. [cite_start]**Explicitly handle code 429** (Too Many Requests) using exponential backoff[cite: 98].
5. Save all matches in the `alignment_mapping.csv` file.

In [16]:
import requests
import time
import csv
from rdflib import Graph, Namespace, URIRef, OWL, RDF, RDFS
from tqdm import tqdm

# ── Setup ─────────────────────────────────────────────────────────────────
EX = Namespace('http://example.org/private#')
WD = Namespace('http://www.wikidata.org/entity/')

g = Graph()
g.parse('initial_private_kb.ttl', format='turtle')
g.bind('wd',  WD)
g.bind('owl', OWL)

HEADERS_API = {
    'User-Agent': 'SemanticWebKBLab/1.0 (university-project; contact: lab@example.com)'
}
WIKIDATA_API = 'https://www.wikidata.org/w/api.php'

# ── Recherche avec retry 429 ───────────────────────────────────────────────
def search_wikidata_entity(label: str, retries: int = 3):
    params = {
        'action': 'wbsearchentities',
        'format': 'json',
        'language': 'en',
        'search': label,
        'limit': 1
    }
    for attempt in range(retries):
        try:
            r = requests.get(WIKIDATA_API, params=params,
                             headers=HEADERS_API, timeout=15)
            if r.status_code == 429:
                wait = 30 * (2 ** attempt)
                tqdm.write(f'  429 Too Many Requests — attente {wait}s...')
                time.sleep(wait)
                continue
            r.raise_for_status()
            data = r.json()
            if data.get('search'):
                return data['search'][0]['id'], 0.95
        except Exception as e:
            tqdm.write(f'  Erreur API ({attempt+1}/{retries}) : {e}')
            time.sleep(5)
    return None, 0.0

# ── Extraction des entites a aligner ──────────────────────────────────────
entities_to_link = {}
for entity_class in [EX.Researcher, EX.Organization, EX.Award]:
    for s, _, _ in g.triples((None, RDF.type, entity_class)):
        lbl = list(g.triples((s, RDFS.label, None)))
        label = str(lbl[0][2]) if lbl else str(s).split('#')[-1].replace('_', ' ')
        entities_to_link[s] = label

print(f'{len(entities_to_link)} entites a aligner...')

# ── Alignement ────────────────────────────────────────────────────────────
mapping_table = []
for private_uri, label in tqdm(entities_to_link.items(), desc='Entity Linking'):
    wd_id, confidence = search_wikidata_entity(label)
    if wd_id:
        ext_uri = WD[wd_id]
        g.add((private_uri, OWL.sameAs, ext_uri))
        mapping_table.append([str(private_uri), str(ext_uri), confidence])
    time.sleep(0.5)

# ── Export ────────────────────────────────────────────────────────────────
with open('alignment_mapping.csv', 'w', newline='', encoding='utf-8') as f:
    writer = csv.writer(f)
    writer.writerow(['Private Entity', 'External URI', 'Confidence'])
    writer.writerows(mapping_table)

g.serialize(destination='aligned_private_kb.ttl', format='turtle')
print(f'\n{len(mapping_table)}/{len(entities_to_link)} entites alignees vers Wikidata')
print('-> alignment_mapping.csv')
print('-> aligned_private_kb.ttl')


77 entites a aligner...


Entity Linking: 100%|██████████████████████████████████████████████████████████████████| 77/77 [01:01<00:00,  1.26it/s]


76/77 entites alignees vers Wikidata
-> alignment_mapping.csv
-> aligned_private_kb.ttl


## Step 3 — Predicate Alignment

**Why this step?**
Properties from our private KB (e.g., `ex:wonAward`, `ex:affiliatedWith`) are linked to their Wikidata equivalents via `owl:equivalentProperty`. [cite_start]This alignment allows for cross-source inferences and standardized data representation[cite: 2].

**Strategy — Pre-validated `PREDICATE_ALIGNMENTS` dictionary**:
Instead of querying the SPARQL endpoint for every individual property—which is slow and prone to false positives—we use a mapping constructed after manual validation. [cite_start]This approach is more robust, faster, and avoids unnecessary API requests[cite: 2].

In [17]:
from rdflib import Graph, Namespace, OWL
from rdflib.namespace import RDFS

# ── Setup ─────────────────────────────────────────────────────────────────
EX  = Namespace('http://example.org/private#')
WDT = Namespace('http://www.wikidata.org/prop/direct/')

g = Graph()
g.parse('aligned_private_kb.ttl', format='turtle')
g.bind('wdt', WDT)
g.bind('owl', OWL)

# ── Dictionnaire de mapping pre-valide ────────────────────────────────────
# Source : Wikidata Property Explorer (https://prop-explorer.toolforge.org/)
PREDICATE_ALIGNMENTS = {
    EX.wonAward:       WDT.P166,   # award received
    EX.affiliatedWith: WDT.P1416,  # affiliation
    EX.employedBy:     WDT.P108,   # employer
    EX.fieldOfStudy:   WDT.P101,   # field of work
    EX.nationality:    WDT.P27,    # country of citizenship
}

print('Alignement des predicats (owl:equivalentProperty) :')
for local_prop, wd_prop in PREDICATE_ALIGNMENTS.items():
    g.add((local_prop, OWL.equivalentProperty, wd_prop))
    local_name = str(local_prop).split('#')[-1]
    wd_id      = str(wd_prop).split('/')[-1]
    print(f'  ex:{local_name:<22} =  wdt:{wd_id}')

g.serialize(destination='fully_aligned_kb.ttl', format='turtle')
print('\n-> fully_aligned_kb.ttl sauvegarde.')


Alignement des predicats (owl:equivalentProperty) :
  ex:wonAward               =  wdt:P166
  ex:affiliatedWith         =  wdt:P1416
  ex:employedBy             =  wdt:P108
  ex:fieldOfStudy           =  wdt:P101
  ex:nationality            =  wdt:P27

-> fully_aligned_kb.ttl sauvegarde.


## Step 4 — KB Expansion via SPARQL (Hops Method)

**Why this step?**
We enrich our KB by retrieving Wikidata triples around the aligned entities and then around their neighbors using a controlled Breadth-First Search (BFS).

**Expansion Architecture**:
* **Pass 1**: Hop 1 from aligned entities (`owl:sameAs`).
* **Pass 2**: Hop 2 from new Wikidata neighbors discovered in Pass 1.

**Control Mechanisms (compliant with PDF guidelines)**:

| Mechanism | Role |
|---|---|
| **Batching** `VALUES ?s {...}` in lots of 50 | [cite_start]Avoids API timeouts[cite: 2]. |
| **Retry 429** with exponential backoff | [cite_start]Resists Wikidata rate-limiting[cite: 2]. |
| **`allowed_preds`** capped at 200 | Guarantees ≤ 200 distinct relations. |
| **`MAX_NEIGHBORS`** = 2000 | [cite_start]Controls combinatorial explosion during Pass 2[cite: 2]. |

In [18]:
import requests
import time
from rdflib import Graph, Namespace, URIRef, OWL
from tqdm import tqdm

# ── Configuration ─────────────────────────────────────────────────────────
EX  = Namespace('http://example.org/private#')
WD  = Namespace('http://www.wikidata.org/entity/')
WDT = Namespace('http://www.wikidata.org/prop/direct/')

g = Graph()
g.parse('fully_aligned_kb.ttl', format='turtle')
g.bind('wd',  WD)
g.bind('wdt', WDT)
g.bind('ex',  EX)

SPARQL_ENDPOINT = 'https://query.wikidata.org/sparql'
HEADERS_SPARQL  = {
    'User-Agent': 'SemanticWebKBLab/1.0 (university-project; contact: lab@example.com)',
    'Accept':     'application/sparql-results+json'
}

# ── Parametres de controle du volume ──────────────────────────────────────
MAX_PREDICATES         = 185   # Plafond strict (consignes PDF)
MAX_NEIGHBORS          = 2500  # Voisins max explores en Passe 2
BATCH_SIZE             = 50    # Entites par requete SPARQL
MAX_TRIPLES_PER_ENTITY = 200   # Limite de bruit par entite

# ── Registre des predicats autorises ──────────────────────────────────────
allowed_preds = set(
    str(p) for s, p, o in g
    if str(p).startswith('http://www.wikidata.org/prop/direct/')
)
print(f'Predicats initiaux (alignements Etape 3) : {len(allowed_preds)}')

# ═════════════════════════════════════════════════════════════════════════
# FONCTIONS UTILITAIRES
# ═════════════════════════════════════════════════════════════════════════

def run_sparql(query: str, retries: int = 4) -> list:
    for attempt in range(retries):
        try:
            r = requests.get(
                SPARQL_ENDPOINT, params={'query': query},
                headers=HEADERS_SPARQL, timeout=60
            )
            if r.status_code == 429:
                wait = 30 * (2 ** attempt)
                tqdm.write(f'  429 Rate Limit — attente {wait}s (tentative {attempt+1}/{retries})')
                time.sleep(wait)
                continue
            r.raise_for_status()
            return r.json()['results']['bindings']
        except Exception as e:
            tqdm.write(f'  Erreur SPARQL ({attempt+1}/{retries}) : {e}')
            time.sleep(10 * (attempt + 1))
    return []


def add_triple_controlled(s: URIRef, p: URIRef, o: URIRef) -> bool:
    p_str = str(p)
    if p_str not in allowed_preds:
        if len(allowed_preds) >= MAX_PREDICATES:
            return False  # Budget epuise
        allowed_preds.add(p_str)
    g.add((s, p, o))
    return True


def expand_entities_batch(wd_ids: list, desc: str = 'Batches') -> int:
    new_triples = 0
    chunks = [wd_ids[i:i+BATCH_SIZE] for i in range(0, len(wd_ids), BATCH_SIZE)]
    for chunk in tqdm(chunks, desc=f'  {desc}', leave=False):
        values_block = ' '.join(f'wd:{eid}' for eid in chunk)
        query = f"""
        SELECT ?s ?p ?o WHERE {{
          VALUES ?s {{ {values_block} }}
          ?s ?p ?o .
          FILTER(STRSTARTS(STR(?p), "http://www.wikidata.org/prop/direct/"))
          FILTER(isIRI(?o))
        }}
        LIMIT {MAX_TRIPLES_PER_ENTITY * len(chunk)}
        """
        results = run_sparql(query)
        for row in results:
            if add_triple_controlled(
                URIRef(row['s']['value']),
                URIRef(row['p']['value']),
                URIRef(row['o']['value'])
            ):
                new_triples += 1
        time.sleep(1.5)
    return new_triples


# ═════════════════════════════════════════════════════════════════════════
# PASSE 1 : HOP 1 — Entites alignees (owl:sameAs)
# ═════════════════════════════════════════════════════════════════════════
print('\n' + '='*60)
print('PASSE 1 — Hop 1 : Entites alignees (owl:sameAs)')
print('='*60)

aligned_ids = [
    str(o).split('/')[-1]
    for s, p, o in g.triples((None, OWL.sameAs, None))
    if 'wikidata.org' in str(o)
]
print(f'Entites de depart : {len(aligned_ids)}')

n_pass1 = expand_entities_batch(aligned_ids, desc='Hop 1')
g.serialize(destination='kb_after_hop1.ttl', format='turtle')
print(f'\nPasse 1 terminee')
print(f'  Triplets ajoutes   : {n_pass1:,}')
print(f'  Triplets totaux    : {len(g):,}')
print(f'  Predicats utilises : {len(allowed_preds)}/{MAX_PREDICATES}')

# ═════════════════════════════════════════════════════════════════════════
# PASSE 2 : HOP 2 — Nouveaux voisins Wikidata
# ═════════════════════════════════════════════════════════════════════════
print('\n' + '='*60)
print('PASSE 2 — Hop 2 : Voisins decouverts a la Passe 1')
print('='*60)

hop2_candidates = set()
for s, p, o in g:
    o_str = str(o)
    s_str = str(s)
    if (o_str.startswith('http://www.wikidata.org/entity/Q')
            and s_str.startswith('http://www.wikidata.org/entity/Q')):
        hop2_candidates.add(o_str.split('/')[-1])

hop2_candidates -= set(aligned_ids)
hop2_ids = list(hop2_candidates)[:MAX_NEIGHBORS]
print(f'Voisins candidats : {len(hop2_candidates):,} -> limite a {len(hop2_ids):,}')

n_pass2 = expand_entities_batch(hop2_ids, desc='Hop 2')
g.serialize(destination='kb_after_hop2.ttl', format='turtle')
print(f'\nPasse 2 terminee')
print(f'  Triplets ajoutes   : {n_pass2:,}')
print(f'  Triplets totaux    : {len(g):,}')
print(f'  Predicats utilises : {len(allowed_preds)}/{MAX_PREDICATES}')

# Export final
g.serialize(destination='extended_kb.nt',  format='nt')
g.serialize(destination='extended_kb.ttl', format='turtle')
print('\nKB etendue sauvegardee -> extended_kb.nt / extended_kb.ttl')


Predicats initiaux (alignements Etape 3) : 0

PASSE 1 — Hop 1 : Entites alignees (owl:sameAs)
Entites de depart : 76



Passe 1 terminee
  Triplets ajoutes   : 5,689
  Triplets totaux    : 6,100
  Predicats utilises : 185/185

PASSE 2 — Hop 2 : Voisins decouverts a la Passe 1
Voisins candidats : 3,283 -> limite a 2,500



Passe 2 terminee
  Triplets ajoutes   : 46,616
  Triplets totaux    : 52,716
  Predicats utilises : 185/185

KB etendue sauvegardee -> extended_kb.nt / extended_kb.ttl


## Step 5 — Final Assessment and Automatic Validation

Verification of the extended graph metrics against the thresholds defined in the PDF guidelines. 
[cite_start]Each line displays **[v] OK** if the value is within the specified range, and **[x] NOT OK** otherwise[cite: 2].

In [19]:
from rdflib import Graph

g_final = Graph()
g_final.parse('extended_kb.nt', format='nt')

# ── Metriques ────────────────────────────────────────────────────────────
total_triplets  = len(g_final)
all_nodes       = set(g_final.subjects()) | set(g_final.objects())
total_entities  = len(all_nodes)
total_relations = len(set(p for s, p, o in g_final))
wd_entities     = len(set(
    str(s) for s, p, o in g_final
    if str(s).startswith('http://www.wikidata.org/entity/Q')
))

# ── Seuils consignes PDF ─────────────────────────────────────────────────
THRESHOLDS = [
    ('Triplets',   total_triplets,  50_000,  200_000),
    ('Entites',    total_entities,   5_000,   30_000),
    ('Relations',  total_relations,     50,      200),
]

SEP = '=' * 68
print(SEP)
print(f"{'BILAN FINAL — KNOWLEDGE BASE ETENDUE':^68}")
print(SEP)
print(f"{'Metrique':<16} {'Valeur':>10}   {'Min':>8}   {'Max':>8}   Statut")
print('-' * 68)

all_ok = True
for label, value, lo, hi in THRESHOLDS:
    ok     = lo <= value <= hi
    mark   = 'v' if ok else 'x'
    status = 'OK' if ok else 'NOT OK'
    if not ok: all_ok = False
    print(f'[{mark}] {label:<14} {value:>10,}   {lo:>8,}   {hi:>8,}   {status}')

print('-' * 68)
print(f'    Entites Wikidata (sujets wdt:) : {wd_entities:,}')
print(SEP)
mark   = 'v' if all_ok else 'x'
result = 'CONFORME AUX CONSIGNES' if all_ok else 'NON CONFORME — voir metriques NOT OK ci-dessus'
print(f'[{mark}] RESULTAT GLOBAL : {result}')
print(SEP)


                BILAN FINAL — KNOWLEDGE BASE ETENDUE                
Metrique             Valeur        Min        Max   Statut
--------------------------------------------------------------------
[v] Triplets           52,716     50,000    200,000   OK
[v] Entites            26,266      5,000     30,000   OK
[v] Relations             196         50        200   OK
--------------------------------------------------------------------
    Entites Wikidata (sujets wdt:) : 2,560
[v] RESULTAT GLOBAL : CONFORME AUX CONSIGNES


## Step 6 — Final Exports

Separate export of the four deliverables required by the lab guidelines:
1. **`extended_kb.nt`** — Complete extended KB in N-Triples format.
2. **`ontology.ttl`** — The TBox schema only, provided in readable Turtle format.
3. **`alignment_mapping.csv`** — Mapping table showing correspondences between private entities and Wikidata.
4. **`stats_final.txt`** — Reproducible textual summary of the final Knowledge Base statistics.

In [20]:
import os
from rdflib import Graph, OWL
from rdflib.namespace import RDFS

# Export 1 : KB etendue (deja generee a l'Etape 4)
size_nt = os.path.getsize('extended_kb.nt') / 1024
print(f'[v] extended_kb.nt          ({size_nt:.0f} KB)')

# Export 2 : Ontologie seule (TBox)
g_onto = Graph()
g_onto.parse('fully_aligned_kb.ttl', format='turtle')

schema_preds = {OWL.Class, OWL.ObjectProperty, OWL.equivalentProperty,
                RDFS.subClassOf, RDFS.domain, RDFS.range}
g_tbox = Graph()
for s, p, o in g_onto:
    if p in schema_preds or o in schema_preds:
        g_tbox.add((s, p, o))

g_tbox.serialize(destination='ontology.ttl', format='turtle')
size_onto = os.path.getsize('ontology.ttl') / 1024
print(f'[v] ontology.ttl            ({size_onto:.1f} KB)')

# Export 3 : Alignements
size_csv = os.path.getsize('alignment_mapping.csv') / 1024
print(f'[v] alignment_mapping.csv   ({size_csv:.1f} KB)')

# Export 4 : Statistiques
g_final = Graph()
g_final.parse('extended_kb.nt', format='nt')
t = len(g_final)
r = len(set(p for s, p, o in g_final))
e = len(set(g_final.subjects()) | set(g_final.objects()))

stats_lines = [
    '=== STATISTIQUES FINALES — KB ETENDUE ===',
    f'Triplets totaux      : {t:,}',
    f'Entites (noeuds)     : {e:,}',
    f'Relations distinctes : {r:,}',
    '',
    '=== CONFORMITE AUX CONSIGNES ===',
    f"Triplets [50k-200k]  : {'OK' if 50_000 <= t <= 200_000 else 'KO'} ({t:,})",
    f"Entites  [5k-30k]    : {'OK' if 5_000  <= e <= 30_000  else 'KO'} ({e:,})",
    f"Relations [50-200]   : {'OK' if 50 <= r <= 200 else 'KO'} ({r:,})",
]
with open('stats_final.txt', 'w', encoding='utf-8') as f:
    f.write('\n'.join(stats_lines))

print('[v] stats_final.txt')
print('\nTous les livrables ont ete generes avec succes.')


[v] extended_kb.nt          (6698 KB)
[v] ontology.ttl            (1.6 KB)
[v] alignment_mapping.csv   (6.7 KB)
[v] stats_final.txt

Tous les livrables ont ete generes avec succes.
